# MLflow autolog

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

# サンプルデータ
X = [[1], [2], [3], [4], [5]]
y = [2, 4, 6, 8, 10]

mlflow.set_experiment("simple-linear-regression")

mlflow.sklearn.autolog()

with mlflow.start_run():
    model = LinearRegression()
    model.fit(X, y)

# mlflow.spark フレーバー

In [ ]:
import mlflow
import mlflow.spark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# サンプルデータ
raw_df = spark.createDataFrame(
    [
        (1.0, 1.0, 2.0),
        (2.0, 1.0, 3.0),
        (3.0, 2.0, 5.0),
        (4.0, 3.0, 7.0),
        (5.0, 3.0, 8.0),
    ],
    ["x1", "x2", "label"],
)
assembler = VectorAssembler(
    inputCols=["x1", "x2"],
    outputCol="features",
)
train_df = (
    assembler.transform(raw_df)
    .select("features", "label")
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=10,
    regParam=0.0,
    elasticNetParam=0.0,
)
model = lr.fit(train_df)

with mlflow.start_run() as run:
    mlflow.log_param("model_class", "pyspark.ml.regression.LinearRegression")
    mlflow.log_metric("num_training_rows", train_df.count())

    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        dfs_tmpdir="Files/spark",
    )

# mlflow.pyspark.ml フレーバー

In [ ]:
import mlflow
import mlflow.spark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# サンプルデータ
raw_df = spark.createDataFrame(
    [
        (1.0, 1.0, 2.0),
        (2.0, 1.0, 3.0),
        (3.0, 2.0, 5.0),
        (4.0, 3.0, 7.0),
        (5.0, 3.0, 8.0),
    ],
    ["x1", "x2", "label"],
)
assembler = VectorAssembler(
    inputCols=["x1", "x2"],
    outputCol="features",
)
train_df = (
    assembler.transform(raw_df)
    .select("features", "label")
)

mlflow.pyspark.ml.autolog(
    log_models=True,
    log_model_signatures=False,  # Vector 型の署名警告を避ける
    exclusive=False,
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=10,
    regParam=0.0,
    elasticNetParam=0.0,
)


with mlflow.start_run() as run:
    model = lr.fit(train_df)